# Embeddings

En este laboratorio veremos cómo funciona un modelo de embedding y su representación gráfica.

## Dependencias

In [1]:
%pip install -q \
  "cohere>=5.13" \
  "openai>=1.50" \
  "sentence-transformers>=3" \
  "umap-learn[plot]" \
  "altair>=5" \
  "datasets>=3" \
  numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 48.6 MB/s eta 0:00:00


## Setup · API keys

Necesitamos dos: **Cohere** (para embeddings multilingual de calidad) y **OpenAI** (para la comparativa).

In [2]:
import os, getpass, numpy as np, pandas as pd, itertools

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key: ")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key (para comparativa): ")

import cohere
co = cohere.Client(os.environ["COHERE_API_KEY"])

def cos_sim(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

Cohere API Key: ··········
OpenAI API Key (para comparativa): ··········


## 1. Similitud entre frases

Embebemos 4 frases y miramos qué tan parecidas son entre sí. La frase 1 y la 4 deberían ser casi idénticas semánticamente, aunque las palabras concretas cambien.

In [3]:
phrases = [
    "El Anillo Único está en Mordor",
    "Me gusta el caldo caliente",
    "El Anillo controla a todos los demás",
    "El Anillo Único se encuentra en Mordor",
]

res = co.embed(
    texts=phrases,
    model="embed-multilingual-v3.0",
    input_type="search_query",
    embedding_types=["float"],
)
embs = res.embeddings.float

print("Similitud coseno entre cada par:\n")
for i, j in itertools.combinations(range(len(phrases)), 2):
    s = cos_sim(embs[i], embs[j])
    short_i = phrases[i][:35] + "..." if len(phrases[i]) > 35 else phrases[i]
    short_j = phrases[j][:35] + "..." if len(phrases[j]) > 35 else phrases[j]
    print(f"  {short_i:38s} vs  {short_j:38s} : {s:.3f}")

Similitud coseno entre cada par:

  El Anillo Único está en Mordor         vs  Me gusta el caldo caliente             : 0.395
  El Anillo Único está en Mordor         vs  El Anillo controla a todos los demá... : 0.678
  El Anillo Único está en Mordor         vs  El Anillo Único se encuentra en Mor... : 0.985
  Me gusta el caldo caliente             vs  El Anillo controla a todos los demá... : 0.426
  Me gusta el caldo caliente             vs  El Anillo Único se encuentra en Mor... : 0.376
  El Anillo controla a todos los demá... vs  El Anillo Único se encuentra en Mor... : 0.650


## 2. Aritmética de vectores · `rey − hombre + mujer ≈ reina`

El truco clásico que descubrieron los autores de Word2Vec en 2013: las **direcciones** del espacio de embeddings codifican relaciones. La dirección de "género" es una de ellas.

Si ese efecto se conserva en los embeddings modernos de frases, deberíamos poder hacer:

$$\vec{rey} - \vec{hombre} + \vec{mujer} \approx \vec{reina}$$

Vamos a probarlo.

In [4]:
vocab = ["rey", "reina", "hombre", "mujer", "príncipe", "princesa", "humano", "persona", "caballo", "silla"]

res = co.embed(
    texts=vocab,
    model="embed-multilingual-v3.0",
    input_type="search_document",
    embedding_types=["float"],
)
emb_map = {w: np.array(v) for w, v in zip(vocab, res.embeddings.float)}

target = emb_map["rey"] - emb_map["hombre"] + emb_map["mujer"]

candidates = [w for w in vocab if w not in {"rey", "hombre", "mujer"}]
ranking = sorted(
    [(w, cos_sim(target, emb_map[w])) for w in candidates],
    key=lambda x: -x[1],
)

print("rey − hombre + mujer ≈ ?  (ordenado por similitud al vector resultante)\n")
for w, s in ranking:
    marker = "  ← TOP" if s == ranking[0][1] else ""
    print(f"  {w:12s} : {s:.3f}{marker}")

rey − hombre + mujer ≈ ?  (ordenado por similitud al vector resultante)

  reina        : 0.836  ← TOP
  princesa     : 0.770
  príncipe     : 0.675
  silla        : 0.669
  caballo      : 0.651
  persona      : 0.629
  humano       : 0.577


## 3. Multilingual · misma frase en 4 idiomas

Un buen modelo multilingual coloca traducciones de la misma frase **cerca** en el espacio vectorial, aunque los caracteres no se parezcan en nada. Esto es lo que hace posible un RAG que indexa contenido en cualquier idioma y responde en otro.

In [5]:
sentences = {
    "es": "El gato duerme en el sofá",
    "en": "The cat sleeps on the couch",
    "fr": "Le chat dort sur le canapé",
    "ja": "猫はソファで寝ています",
}
distractor = "Una receta de paella valenciana"

all_texts = list(sentences.values()) + [distractor]
res = co.embed(
    texts=all_texts,
    model="embed-multilingual-v3.0",
    input_type="search_query",
    embedding_types=["float"],
)
embs = {lang: np.array(v) for lang, v in zip(sentences.keys(), res.embeddings.float)}
d_emb = np.array(res.embeddings.float[-1])

print("Similitud entre la MISMA frase en distintos idiomas (debería ser alta):\n")
for a, b in itertools.combinations(sentences.keys(), 2):
    print(f"  {a} ↔ {b} : {cos_sim(embs[a], embs[b]):.3f}")

print(f"\nSimilitud con un DISTRACTOR ('{distractor}', debería ser baja):\n")
for a in sentences.keys():
    print(f"  {a} ↔ distractor : {cos_sim(embs[a], d_emb):.3f}")

Similitud entre la MISMA frase en distintos idiomas (debería ser alta):

  es ↔ en : 0.812
  es ↔ fr : 0.853
  es ↔ ja : 0.858
  en ↔ fr : 0.801
  en ↔ ja : 0.794
  fr ↔ ja : 0.805

Similitud con un DISTRACTOR ('Una receta de paella valenciana', debería ser baja):

  es ↔ distractor : 0.429
  en ↔ distractor : 0.237
  fr ↔ distractor : 0.392
  ja ↔ distractor : 0.310


## 4. Comparativa de modelos · OpenAI vs Cohere vs BGE-M3

Tres modelos de embedding muy distintos:

| Modelo | Tipo | Dim |
|---|---|---|
| `text-embedding-3-small` | OpenAI cerrado, $0.02 / 1M tokens | 1536 |
| `embed-multilingual-v3.0` | Cohere cerrado, multilingual nativo | 1024 |
| `BAAI/bge-m3` | HuggingFace, **open source**, multilingual | 1024 |

Vamos a embeber las mismas 4 frases con los tres y comparar la similitud entre la pareja español/inglés (texts[0] vs texts[1]) según cada modelo. Esperamos: los tres dan alta similitud, pero los valores absolutos difieren.

In [6]:
texts = [
    "El gato duerme en el sofá",
    "The cat sleeps on the couch",
    "Una receta de tortilla de patatas",
    "Python es un lenguaje de programación",
]

# OpenAI
from openai import OpenAI
oai = OpenAI()
oai_embs = np.array([d.embedding for d in oai.embeddings.create(model="text-embedding-3-small", input=texts).data])

# Cohere
co_res = co.embed(texts=texts, model="embed-multilingual-v3.0", input_type="search_query", embedding_types=["float"])
co_embs = np.array(co_res.embeddings.float)

# HuggingFace BGE-M3 (open, primer load descarga ~2GB)
from sentence_transformers import SentenceTransformer
bge = SentenceTransformer("BAAI/bge-m3")
bge_embs = bge.encode(texts, normalize_embeddings=True)

print("Dimensiones del vector:")
print(f"  OpenAI text-embedding-3-small : {oai_embs.shape[1]}")
print(f"  Cohere embed-multilingual-v3  : {co_embs.shape[1]}")
print(f"  BGE-M3 (open)                 : {bge_embs.shape[1]}")

print("\nSimilitud entre la frase ES y EN del gato (texts[0] vs texts[1]):")
for name, emb in [("OpenAI", oai_embs), ("Cohere", co_embs), ("BGE-M3", bge_embs)]:
    print(f"  {name:10s} : {cos_sim(emb[0], emb[1]):.3f}")

print("\nLos 3 modelos coinciden en que las dos frases son semánticamente cercanas,")
print("aunque la dimensión y la escala absoluta difieren. Son intercambiables.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dimensiones del vector:
  OpenAI text-embedding-3-small : 1536
  Cohere embed-multilingual-v3  : 1024
  BGE-M3 (open)                 : 1024

Similitud entre la frase ES y EN del gato (texts[0] vs texts[1]):
  OpenAI     : 0.688
  Cohere     : 0.812
  BGE-M3     : 0.764

Los 3 modelos coinciden en que las dos frases son semánticamente cercanas,
aunque la dimensión y la escala absoluta difieren. Son intercambiables.


## 5. Visualización con UMAP

Embebemos varias frases (preguntas + respuestas + distractores) y las proyectamos a 2D con UMAP para ver cómo se agrupan. Las parejas pregunta/respuesta deberían quedar cerca.

In [7]:
sentences_df = pd.DataFrame({"text": [
    "¿Dónde está el Anillo Único?",
    "El Anillo Único está en Mordor",

    "¿Dónde viven los Hobbits?",
    "Los Hobbits viven en La Comarca",

    "¿Qué es el lembas?",
    "El lembas es el pan élfico para viajes largos",

    "¿Cuál es la capital de Francia?",
    "La capital de Francia es París",
]})

emb = co.embed(
    texts=list(sentences_df["text"]),
    model="embed-multilingual-v3.0",
    input_type="search_document",
    embedding_types=["float"],
).embeddings.float

import umap, altair as alt
reducer = umap.UMAP(n_neighbors=2, random_state=42)
proj = reducer.fit_transform(np.array(emb))
df_plot = sentences_df.copy()
df_plot["x"], df_plot["y"] = proj[:, 0], proj[:, 1]

chart = (
    alt.Chart(df_plot)
    .mark_circle(size=80)
    .encode(
        x=alt.X("x", scale=alt.Scale(zero=False)),
        y=alt.Y("y", scale=alt.Scale(zero=False)),
        tooltip=["text"],
    )
    .properties(width=600, height=400, title="frases en espacio semántico (UMAP 2D)")
)
chart.interactive()

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


alt.Chart(...)

## 6. Embeddings más grandes · Wikipedia

Cargamos 5000 artículos de Wikipedia ya pre-embebidos por Cohere (dataset `Cohere/wikipedia-2023-11-embed-multilingual-v3`) y los visualizamos.

In [8]:
from datasets import load_dataset

lang = "simple"
max_docs = 5000
docs_stream = load_dataset("Cohere/wikipedia-2023-11-embed-multilingual-v3", lang, split="train", streaming=True)

titles, texts_wiki, doc_embeddings = [], [], []
for doc in docs_stream:
    titles.append(doc["title"])
    texts_wiki.append(doc["text"])
    doc_embeddings.append(doc["emb"])
    if len(titles) >= max_docs:
        break

wiki_df = pd.DataFrame({"title": titles})
embs_arr = np.array(doc_embeddings)
print(f"cargados {len(titles)} docs · embedding shape: {embs_arr.shape}")

README.md:   0%|          | 0.00/30.2k [00:00<?, ?B/s]

cargados 5000 docs · embedding shape: (5000, 1024)


In [9]:
reducer_big = umap.UMAP(n_neighbors=100, random_state=42)
proj_big = reducer_big.fit_transform(embs_arr)
wiki_df["x"], wiki_df["y"] = proj_big[:, 0], proj_big[:, 1]

(
    alt.Chart(wiki_df)
    .mark_circle(size=20, opacity=0.5)
    .encode(
        x=alt.X("x", scale=alt.Scale(zero=False)),
        y=alt.Y("y", scale=alt.Scale(zero=False)),
        tooltip=["title"],
    )
    .properties(width=800, height=600, title=f"Wikipedia '{lang}' · {max_docs} docs en espacio semántico 2D")
    .interactive()
)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


alt.Chart(...)

## 7. Búsqueda semántica · query en español sobre wiki en inglés

Como Cohere `embed-multilingual` proyecta cualquier idioma al mismo espacio, podemos hacer una query en **español** contra un corpus en **inglés** y encontrar resultados. El mismo truco que hace posible buscar en Wikipedia japonesa con una pregunta en castellano.

In [10]:
query = "¿Cuántos habitantes tiene España?"

q_emb = np.array(
    co.embed(
        texts=[query],
        model="embed-multilingual-v3.0",
        input_type="search_query",
        embedding_types=["float"],
    ).embeddings.float[0]
)

# Los embeddings de Cohere ya vienen normalizados, así que el producto escalar = similitud coseno
scores = embs_arr @ q_emb
top_k = 5
top_idx = np.argsort(-scores)[:top_k]

print(f"Query: {query}\n")
for rank, idx in enumerate(top_idx, 1):
    print(f"#{rank} · {titles[idx]} · score {scores[idx]:.3f}")
    print(f"    {texts_wiki[idx][:200]}...\n")

Query: ¿Cuántos habitantes tiene España?

#1 · France · score 0.569
    On 1 January 2008, it was estimated that 63.8 million people live in France, including in the Overseas Regions of France. 61,875,000 of these live in metropolitan France, the part of the country that ...

#2 · United Kingdom · score 0.557
    About 68 million people live in the UK (2022). They can be divided into four big nationalities based on the countries where they live (or where they were born or their ancestry). Each country has a de...

#3 · Italy · score 0.553
    The population of Italy is about 60 million people. Almost 3 million of them live in Rome, and 1.5 million in Milan. As of December 2015, over 5 million foreigners were living in Italy, which is 8.3% ...

#4 · India · score 0.551
    There are 1.4 billion people living in India. In 2023, India passed China to become the world's most populous country....

#5 · Madrid · score 0.528
    Madrid (pronounced: “mah-drid or /məˈdrɪd/) is the capital and l

## Cierre

Lo que has visto:

- Un embedding es un **vector** que representa significado
- La similitud coseno te dice **cuán parecidos** son dos textos
- Las **direcciones** del espacio capturan relaciones (género, número, concepto)
- Los modelos multilingual cruzan idiomas en el mismo espacio
- OpenAI, Cohere y BGE-M3 son intercambiables · solo cambia la dim

Lo que viene:

- En el siguiente notebook (`Vector Stores.ipynb`) usaremos estos embeddings para indexar y buscar a escala
- En `Embeddings multimodal.ipynb` veremos cómo embeber **imágenes y texto en el mismo espacio**
- En **M19 (RAG)** llegará el pipeline completo: chunking, retrieval con metadata, reranking